In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt
import statsmodels.api as sm
from tqdm import tqdm


In [3]:
path = '..\\data\\segment_alerts_all_airports_train.csv'
df = pd.read_csv(path)
df.head()

,lightning_id,lightning_airport_id,date,lon,lat,amplitude,maxis,icloud,dist,azimuth,airport,airport_alert_id,is_last_lightning_cloud_ground
0,1,1,2016-01-02 14:53:36+00:00,9.0559,42.0826,-9.90,0.3,False,27.360653,57.852343,Ajaccio,NaN,NaN
1,2,2,2016-01-02 14:53:36+00:00,9.0236,42.0953,-3.33,0.2,True,26.383167,52.117828,Ajaccio,NaN,NaN
2,3,3,2016-01-02 21:22:53+00:00,8.8585,42.0456,-18.68,0.4,True,14.313391,24.500543,Ajaccio,NaN,NaN
3,4,4,2016-01-02 21:22:53+00:00,8.8517,42.0517,-7.51,0.2,False,14.794117,20.854458,Ajaccio,1.0,False
4,5,5,2016-01-02 21:24:46+00:00,8.8728,42.0494,-6.01,0.2,False,15.124224,29.058471,Ajaccio,1.0,False


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 507071 entries, 0 to 507070
Data columns (total 13 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   lightning_id                    507071 non-null  int64  
 1   lightning_airport_id            507071 non-null  int64  
 2   date                            507071 non-null  object 
 3   lon                             507071 non-null  float64
 4   lat                             507071 non-null  float64
 5   amplitude                       507071 non-null  float64
 6   maxis                           507071 non-null  float64
 7   icloud                          507071 non-null  bool   
 8   dist                            507071 non-null  float64
 9   azimuth                         507071 non-null  float64
 10  airport                         507071 non-null  object 
 11  airport_alert_id                56599 non-null   float64
 12  is_last_lightnin

In [4]:
df.groupby(['airport'])['airport_alert_id'].sum()

airport
Ajaccio     2550128.0
Bastia      4048591.0
Biarritz    2734361.0
Nantes       514309.0
Pise        8090686.0
Name: airport_alert_id, dtype: float64

In [18]:
df.tail(20)

,lightning_id,lightning_airport_id,date,lon,lat,amplitude,maxis,icloud,dist,azimuth,airport,airport_alert_id,is_last_lightning_cloud_ground
507051,628707,156699,2022-12-11 07:37:26+00:00,10.5516,43.7487,-8.87,0.098,False,13.630384,70.613010,Pise,769.0,False
507052,628708,156700,2022-12-11 07:41:37+00:00,10.6308,43.7757,44.78,0.180,False,20.658658,70.804696,Pise,NaN,NaN
507053,628709,156701,2022-12-11 07:44:46+00:00,10.6253,43.7809,18.66,0.103,True,20.523323,69.214008,Pise,NaN,NaN
507054,628710,156702,2022-12-11 07:48:43+00:00,10.6547,43.7171,-134.22,4.698,False,20.686146,85.060235,Pise,NaN,NaN
507055,628711,156703,2022-12-11 07:51:45+00:00,10.3073,43.8531,40.62,0.128,False,19.046601,329.885724,Pise,769.0,False
507056,628712,156704,2022-12-11 07:52:50+00:00,10.6413,43.7669,10.04,0.136,True,21.031981,73.472293,Pise,NaN,NaN
507057,628713,156705,2022-12-11 07:57:08+00:00,10.6726,43.7635,14.49,0.131,True,23.250481,75.944049,Pise,NaN,NaN
507058,628714,156706,2022-12-11 07:58:28+00:00,10.6585,43.7692,21.28,0.084,True,22.408087,74.042940,Pise,NaN,NaN
507059,628715,156707,2022-12-11 08:01:15+00:00,10.6806,43.7802,22.19,0.136,True,24.510683,73.166452,Pise,NaN,NaN
507060,628716,156708,2022-12-11 08:02:58+00:00,10.6829,43.7769,14.17,0.099,True,24.543371,73.908109,Pise,NaN,NaN


In [14]:
bol = (df['airport'] == 'Ajaccio') & (df['airport_alert_id'] <= 5)
df.loc[bol, :]

,lightning_id,lightning_airport_id,date,lon,lat,amplitude,maxis,icloud,dist,azimuth,airport,airport_alert_id,is_last_lightning_cloud_ground
3,4,4,2016-01-02 21:22:53+00:00,8.8517,42.0517,-7.51,0.200000,False,14.794117,20.854458,Ajaccio,1.0,False
4,5,5,2016-01-02 21:24:46+00:00,8.8728,42.0494,-6.01,0.200000,False,15.124224,29.058471,Ajaccio,1.0,False
5,6,6,2016-01-02 21:25:59+00:00,8.8495,42.0595,-23.96,2.600000,False,15.583922,18.926800,Ajaccio,1.0,False
6,7,7,2016-01-02 21:27:04+00:00,8.8978,42.0423,-11.65,0.200000,False,15.343433,38.642144,Ajaccio,1.0,False
8,9,9,2016-01-02 21:28:54+00:00,8.9119,42.0593,-5.31,0.600000,False,17.561774,38.772909,Ajaccio,1.0,True
29,30,30,2016-01-12 06:37:45+00:00,8.8867,42.0213,-13.74,0.068709,False,12.875968,40.620607,Ajaccio,2.0,True
38,39,39,2016-01-12 07:25:37+00:00,8.7287,41.9383,-25.34,0.052910,False,6.347720,281.205948,Ajaccio,3.0,True
52,53,53,2016-02-07 14:05:14+00:00,8.8132,41.8974,-24.81,0.050000,False,3.033382,158.538733,Ajaccio,4.0,True
54,55,55,2016-02-07 14:35:39+00:00,8.7617,42.0820,-11.78,0.116000,False,17.927176,345.420363,Ajaccio,5.0,True
